# Notebook 06 — Geometry Sweeps: Electrode Width and Gap

This notebook extends the reduced ion-trap workflow by sweeping basic electrode-geometry parameters and tracking how the pseudopotential landscape changes.

It focuses on:

- rebuilding the lower-boundary electrode geometry across a 2D sweep
- solving Laplace's equation for each geometry
- computing electric fields and normalized pseudopotentials
- locating a candidate interior trap point for each geometry
- extracting local curvature information from the Hessian
- mapping trap height, curvature-based strength, and validity across geometry space

This provides a first geometry-design workflow for comparing surface-style layouts.


## Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve


## Grid and fixed voltages

We keep a fixed computational grid and fixed voltages, and vary only the lower-boundary geometry:

- center electrode width
- gap between the center electrode and each side electrode

The side electrodes are kept symmetric.


In [ ]:
# Domain size and resolution
nx, ny = 121, 81
x = np.linspace(-3.0, 3.0, nx)
y = np.linspace(0.0, 4.0, ny)
dx = x[1] - x[0]
dy = y[1] - y[0]

# Fixed voltages for the geometry sweep
v_rf = 1.0
v_dc = -1.5

# Fixed side-electrode width
side_width = 1.3

print(f'Grid: {nx} x {ny}')
print(f'dx = {dx:.3f}, dy = {dy:.3f}')
print(f'Fixed voltages: v_rf = {v_rf:.2f}, v_dc = {v_dc:.2f}')


## Helper functions

We define helper functions to:

- build a symmetric lower-boundary electrode geometry from width and gap
- solve Laplace's equation
- compute the electric field and normalized pseudopotential
- locate a candidate interior trap point
- compute local Hessian-based curvature diagnostics


In [ ]:
def idx(i, j, nx):
    return i * nx + j

def make_electrode_masks(center_width, gap, side_width):
    center_half = center_width / 2.0
    left_center = -(center_half + gap + side_width / 2.0)
    right_center = +(center_half + gap + side_width / 2.0)

    left_mask = (x >= left_center - side_width / 2.0) & (x <= left_center + side_width / 2.0)
    center_mask = (x >= -center_half) & (x <= center_half)
    right_mask = (x >= right_center - side_width / 2.0) & (x <= right_center + side_width / 2.0)
    remaining_ground = ~(left_mask | center_mask | right_mask)
    return left_mask, center_mask, right_mask, remaining_ground

def build_boundary_problem(left_mask, center_mask, right_mask, remaining_ground, v_rf, v_dc):
    V = np.zeros((ny, nx), dtype=float)
    fixed = np.zeros((ny, nx), dtype=bool)

    # Ground outer boundaries
    V[:, 0] = 0.0
    V[:, -1] = 0.0
    V[-1, :] = 0.0
    fixed[:, 0] = True
    fixed[:, -1] = True
    fixed[-1, :] = True

    # Lower boundary electrodes
    bottom = 0
    V[bottom, left_mask] = v_dc
    V[bottom, center_mask] = v_rf
    V[bottom, right_mask] = v_dc
    V[bottom, remaining_ground] = 0.0
    fixed[bottom, :] = True
    return V, fixed

def solve_laplace(V, fixed):
    N = nx * ny
    A = lil_matrix((N, N), dtype=float)
    b = np.zeros(N, dtype=float)

    cx = 1.0 / dx**2
    cy = 1.0 / dy**2
    c0 = -2.0 * (cx + cy)

    for i in range(ny):
        for j in range(nx):
            k = idx(i, j, nx)

            if fixed[i, j]:
                A[k, k] = 1.0
                b[k] = V[i, j]
            else:
                A[k, idx(i, j, nx)] = c0
                A[k, idx(i, j - 1, nx)] = cx
                A[k, idx(i, j + 1, nx)] = cx
                A[k, idx(i - 1, j, nx)] = cy
                A[k, idx(i + 1, j, nx)] = cy

    sol = spsolve(A.tocsr(), b)
    return sol.reshape((ny, nx))

def compute_pseudopotential(Vsol):
    dV_dy, dV_dx = np.gradient(Vsol, dy, dx)
    Ex = -dV_dx
    Ey = -dV_dy
    E_mag = np.sqrt(Ex**2 + Ey**2)
    Phi_pseudo = E_mag**2
    return Ex, Ey, E_mag, Phi_pseudo

def find_candidate_point(Phi_pseudo):
    margin_i_low = 2
    margin_i_high = 3
    margin_j = 3
    i_slice = slice(margin_i_low, ny - margin_i_high)
    j_slice = slice(margin_j, nx - margin_j)
    Phi_search = Phi_pseudo[i_slice, j_slice]
    min_flat = np.argmin(Phi_search)
    min_i_local, min_j_local = np.unravel_index(min_flat, Phi_search.shape)
    min_i = min_i_local + margin_i_low
    min_j = min_j_local + margin_j
    return min_i, min_j

def second_derivatives(F, i, j):
    d2F_dx2 = (F[i, j + 1] - 2.0 * F[i, j] + F[i, j - 1]) / dx**2
    d2F_dy2 = (F[i + 1, j] - 2.0 * F[i, j] + F[i - 1, j]) / dy**2
    d2F_dxdy = (
        F[i + 1, j + 1] - F[i + 1, j - 1] - F[i - 1, j + 1] + F[i - 1, j - 1]
    ) / (4.0 * dx * dy)
    return d2F_dx2, d2F_dy2, d2F_dxdy

def geometry_metrics(Phi_pseudo):
    i, j = find_candidate_point(Phi_pseudo)
    trap_x = x[j]
    trap_y = y[i]
    trap_value = Phi_pseudo[i, j]

    center = Phi_pseudo[i, j]
    neighbors = np.array([
        Phi_pseudo[i - 1, j],
        Phi_pseudo[i + 1, j],
        Phi_pseudo[i, j - 1],
        Phi_pseudo[i, j + 1],
    ])
    is_local_min = np.all(center <= neighbors)

    d2x, d2y, d2xy = second_derivatives(Phi_pseudo, i, j)
    H = np.array([[d2x, d2xy], [d2xy, d2y]], dtype=float)
    eigvals, eigvecs = np.linalg.eigh(H)

    # Stronger filter than Notebook 05: only count fully confining points
    if np.min(eigvals) > 0 and is_local_min:
        trap_strength = float(np.sum(eigvals))
    else:
        trap_strength = 0.0

    omega_proxy = np.sqrt(np.clip(eigvals, 0.0, None))
    min_curvature = float(np.min(eigvals))

    return {
        'i': i,
        'j': j,
        'trap_x': trap_x,
        'trap_y': trap_y,
        'trap_value': trap_value,
        'is_local_min': bool(is_local_min),
        'H': H,
        'eigvals': eigvals,
        'omega_proxy': omega_proxy,
        'trap_strength': trap_strength,
        'min_curvature': min_curvature,
    }


## Geometry sweep setup

We sweep:

- `center_width`: width of the center RF-like electrode
- `gap`: spacing between the center electrode and each side electrode

and evaluate trap metrics for each pair.


In [ ]:
center_width_values = np.linspace(0.4, 1.4, 6)
gap_values = np.linspace(0.15, 0.70, 6)

trap_height_map = np.zeros((len(gap_values), len(center_width_values)))
trap_strength_map = np.zeros((len(gap_values), len(center_width_values)))
min_curvature_map = np.zeros((len(gap_values), len(center_width_values)))
is_local_min_map = np.zeros((len(gap_values), len(center_width_values)), dtype=bool)

results = {}

print(f'Sweeping {len(gap_values) * len(center_width_values)} geometry pairs...')


## Run the geometry sweep


In [ ]:
for igap, gap in enumerate(gap_values):
    for iw, center_width in enumerate(center_width_values):
        left_mask, center_mask, right_mask, remaining_ground = make_electrode_masks(
            center_width=center_width,
            gap=gap,
            side_width=side_width,
        )

        V, fixed = build_boundary_problem(
            left_mask=left_mask,
            center_mask=center_mask,
            right_mask=right_mask,
            remaining_ground=remaining_ground,
            v_rf=v_rf,
            v_dc=v_dc,
        )

        Vsol = solve_laplace(V, fixed)
        Ex, Ey, E_mag, Phi_pseudo = compute_pseudopotential(Vsol)
        metrics = geometry_metrics(Phi_pseudo)

        trap_height_map[igap, iw] = metrics['trap_y']
        trap_strength_map[igap, iw] = metrics['trap_strength']
        min_curvature_map[igap, iw] = metrics['min_curvature']
        is_local_min_map[igap, iw] = metrics['is_local_min']

        results[(float(gap), float(center_width))] = {
            'left_mask': left_mask,
            'center_mask': center_mask,
            'right_mask': right_mask,
            'remaining_ground': remaining_ground,
            'Vsol': Vsol,
            'Phi_pseudo': Phi_pseudo,
            **metrics,
        }

print('Geometry sweep complete.')


## Trap-height map


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(
    trap_height_map,
    origin='lower',
    aspect='auto',
    extent=[center_width_values[0], center_width_values[-1], gap_values[0], gap_values[-1]],
)
ax.set_title('Candidate Trap Height vs Geometry Pair')
ax.set_xlabel('Center electrode width')
ax.set_ylabel('Gap')
fig.colorbar(im, ax=ax, label='Trap height')
plt.tight_layout()
plt.show()


## Trap-strength map


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(
    trap_strength_map,
    origin='lower',
    aspect='auto',
    extent=[center_width_values[0], center_width_values[-1], gap_values[0], gap_values[-1]],
)
ax.set_title('Curvature-Based Trap Strength vs Geometry Pair')
ax.set_xlabel('Center electrode width')
ax.set_ylabel('Gap')
fig.colorbar(im, ax=ax, label='Trap strength')
plt.tight_layout()
plt.show()


## Minimum-curvature map


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(
    min_curvature_map,
    origin='lower',
    aspect='auto',
    extent=[center_width_values[0], center_width_values[-1], gap_values[0], gap_values[-1]],
)
ax.set_title('Minimum Principal Curvature vs Geometry Pair')
ax.set_xlabel('Center electrode width')
ax.set_ylabel('Gap')
fig.colorbar(im, ax=ax, label='Minimum curvature')
plt.tight_layout()
plt.show()


## Local-minimum validity map


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(
    is_local_min_map.astype(float),
    origin='lower',
    aspect='auto',
    extent=[center_width_values[0], center_width_values[-1], gap_values[0], gap_values[-1]],
    vmin=0,
    vmax=1,
)
ax.set_title('Immediate-Neighbor Local-Minimum Check')
ax.set_xlabel('Center electrode width')
ax.set_ylabel('Gap')
fig.colorbar(im, ax=ax, label='1 = local minimum')
plt.tight_layout()
plt.show()


## Example best configuration by geometry-based trap strength

We inspect one configuration chosen by the maximum curvature-based trap-strength score.


In [ ]:
best_idx = np.unravel_index(np.argmax(trap_strength_map), trap_strength_map.shape)
best_igap, best_iw = best_idx
best_gap = float(gap_values[best_igap])
best_center_width = float(center_width_values[best_iw])
best = results[(best_gap, best_center_width)]

print(f'Best configuration by trap strength: gap = {best_gap:.3f}, center_width = {best_center_width:.3f}')
print(f"Candidate point: x = {best['trap_x']:.3f}, y = {best['trap_y']:.3f}")
print(f"Trap strength: {best['trap_strength']:.3e}")
print(f"Minimum curvature: {best['min_curvature']:.3e}")
print(f"Local minimum check: {best['is_local_min']}")
print('Curvature eigenvalues:')
print(best['eigvals'])
print('Secular-frequency proxies:')
print(best['omega_proxy'])


In [ ]:
X, Y = np.meshgrid(x, y)

fig, ax = plt.subplots(figsize=(9, 5.5))
im = ax.pcolormesh(X, Y, best['Phi_pseudo'], shading='auto')
cs = ax.contour(X, Y, best['Phi_pseudo'], levels=15)
ax.clabel(cs, inline=True, fontsize=8)
ax.scatter([best['trap_x']], [best['trap_y']], s=90, label='Candidate trap point')
ax.plot(x[best['left_mask']], np.full(np.sum(best['left_mask']), y[0]), linewidth=7)
ax.plot(x[best['center_mask']], np.full(np.sum(best['center_mask']), y[0]), linewidth=7)
ax.plot(x[best['right_mask']], np.full(np.sum(best['right_mask']), y[0]), linewidth=7)
ax.set_title('Best Geometry by Curvature-Based Trap Strength')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()
fig.colorbar(im, ax=ax, label='Pseudo')
plt.tight_layout()
plt.show()


## Show the lower-boundary electrode geometry for the best case


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))

ax.fill_between(x[best['left_mask']], -0.12, 0.12, label='Left DC-like')
ax.fill_between(x[best['center_mask']], -0.12, 0.12, label='Center RF-like')
ax.fill_between(x[best['right_mask']], -0.12, 0.12, label='Right DC-like')

ax.set_title('Best Geometry: Lower-Boundary Electrode Layout')
ax.set_xlabel('x')
ax.set_ylim(-0.35, 0.45)
ax.set_yticks([])
ax.set_aspect('auto')
ax.grid(False)
ax.legend(loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.18))
plt.tight_layout()
plt.show()


## Notes and next steps

This notebook adds a basic geometry-sweep workflow to the reduced ion-trap model.

Natural next steps:

- refine the geometry grid near promising regions
- co-sweep voltages and geometry together
- include explicit physical constants for SI-scale frequency estimates
- add axial confinement and compensation fields
- compare multiple objective functions for trap quality
- turn the sweep into a true design optimization loop

That progression will move the workflow toward practical surface-electrode trap design exploration.
